# Rigid body dynamics

Create a rigid body with out a definite shape.

In [2]:
import numpy as np

from pysph.base.kernels import CubicSpline
from pysph.base.utils import (get_particle_array_rigid_body,
                              get_particle_array)
from pysph.sph.integrator import EPECIntegrator
from pysph.solver.application import Application
from pysph.sph.scheme import add_bool_argument
from pysph.sph.scheme import SchemeChooser
from rigid_body import (setup_rigid_body_unconstrained_dynamics,
                        setup_rigid_body_collision_dynamics,
                        RigidBodyScheme)
from rigid_body_setup import (set_angular_momentum)
from pysph.examples.solid_mech.impact import add_properties
from pysph.tools.geometry import get_3d_block, show_3d



## Create a rigid body and test its properties


In [4]:
from pysph.tools.geometry import get_3d_block, show_3d

from rigid_body_setup import (
    set_total_mass,
    set_center_of_mass,
    set_moment_of_inertia,
    set_mi_in_body_frame_optimized,
    set_body_frame_position_vectors)


def create_an_unsymmetric_body(dx, l, b, h):
    # create a rigid body
    x, y, z = get_3d_block(dx, l, b, h, center=[l/2., b/2., h/2.])
    fltr = (x > l/4.) & (y < b/4.) & (z < h/6.)

    x = x[~fltr]
    y = y[~fltr]
    z = z[~fltr]

    return x, y, z

dx = 0.1
l = 1
b = 2
h = 4
x, y, z = create_an_unsymmetric_body(dx, l, b, h)

hdx = 1.3
rho0 = 1000
m = np.ones_like(x) * dx * dx * dx * rho0
h = np.ones_like(x) * hdx * dx
# radius of each sphere constituting in cube
rad_s = np.ones_like(x) * dx/4.


body = get_particle_array(name='body', x=x, y=y, z=z, h=h, m=m)

body_id = np.zeros(len(x), dtype=int)
body.add_property('body_id', type='int')
body.body_id[:] = body_id[:]


body_id = body.body_id

nb = np.max(body_id) + 1

# first add all the rigid body properties
# Every other rigid body scheme or implementation must
# have to be based on these following properties
body.add_constant("total_mass", np.zeros(nb))
# moment of inetria inverse in body frame
body.add_constant("mib", np.zeros(9*nb))
# moment of inetria inverse in global frame
body.add_constant("mig", np.zeros(9*nb))
# moment of inetria in global frame
body.add_constant("moig", np.zeros(9*nb))

# moment of inetria inverse in princibodyl body frame
body.add_constant("mibp", np.zeros(3*nb))

body.add_constant("cm", np.zeros(3*nb))
body.add_constant("vc", np.zeros(3*nb))
body.add_constant("omega", np.zeros(3*nb))
body.add_constant("ang_mom", np.zeros(3*nb))
body.add_constant("force", np.zeros(3*nb))
body.add_constant("torque", np.zeros(3*nb))

body.add_constant("cm0", np.zeros(3*nb))
body.add_constant("vc0", np.zeros(3*nb))
body.add_constant("omega0", np.zeros(3*nb))
body.add_constant("ang_mom0", np.zeros(3*nb))

# position of bodyrticles in local frame
body.add_property('dx0')
body.add_property('dy0')
body.add_property('dz0')

# total no of rigid bodies
body.add_constant("nb", nb)

# if the rigid body uses DCM then we need following property
body.add_constant("R", [1., 0., 0., 0., 1., 0., 0., 0., 1.] * nb)
body.add_constant("R0", [1., 0., 0., 0., 1., 0., 0., 0., 1.] * nb)

# if the rigid body uses quaternion
body.add_constant("q", [1., 0., 0., 0.] * nb)
body.add_constant("q0", [1., 0., 0., 0.] * nb)

# Find total mass
set_total_mass(body)
set_center_of_mass(body)

## Check rigid body dynamics with DCM and solving with angular momentum

We need the following properties to be precomputed. 

1. Total mass
2. Moment of inertia in body frame

## Tests

1. We will rotate the body by some rotation matrix. Then compute the moment of inertia using body moi and by direct computation.

In [5]:
np.testing.assert_allclose(body.mib[:], np.zeros(9))

In [6]:
set_moment_of_inertia(body)

In [7]:
body.mib.reshape(3, 3)

array([[ 6.37127542e-05,  1.32961116e-07,  1.10150155e-06],
       [ 1.32961116e-07,  7.51675209e-05, -7.00655353e-06],
       [ 1.10150155e-06, -7.00655353e-06,  2.38295461e-04]])

In [8]:
body.R

array([1., 0., 0., 0., 1., 0., 0., 0., 1.])

In [9]:
set_mi_in_body_frame_optimized(body)

In [10]:
body.mibp

array([2.38602701e-04, 7.48700363e-05, 6.37029982e-05])

In [11]:
body.R

array([ 0.0062598 , -0.01612205, -0.99985044, -0.04282527, -0.99895701,
        0.01583953,  0.99906297, -0.04271971,  0.0069437 ])

In [14]:
# here we test the moment of inertia in pricipal axis of body fixed frame
R = body.R.reshape(3, 3)
mibp = np.array([[body.mibp[0], 0., 0.], [0., body.mibp[1], 0.], [0., 0., body.mibp[2]]])

mib = body.mib.reshape(3, 3)


np.testing.assert_almost_equal(np.matmul(R, np.matmul(mibp, R.transpose())), mib)                
                 